# Nano-U: 4-Fold Leave-One-Scene-Out (LOSO) Training on GPU
**Master's Thesis:** *Extending IoT-Robotics Convergence to Learned Perception*
**Author:** Tharen Candi (IAS-Lab / VIMP-Lab, University of Padua)

> **Prerequisites:**
> 1. Set runtime to **T4 GPU** ( →  → ).
> 2. Set your GitHub token in Colab Secrets: Click the **🔑 (Secrets)** tab in the left sidebar, add , and toggle **Notebook access** ON.
> 3. Ensure  is uploaded to your Google Drive root.

In [ ]:
# 1. Authenticate with GITHUB_TOKEN & Clone Fork
import os
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/tharencandi/Nano-U.git"
    print("✓ Authenticated via GITHUB_TOKEN from Colab Secrets.")
except Exception as e:
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        repo_url = f"https://{token}@github.com/tharencandi/Nano-U.git"
        print("✓ Authenticated via GITHUB_TOKEN from environment.")
    else:
        repo_url = "https://github.com/tharencandi/Nano-U.git"
        print("ℹ GITHUB_TOKEN not found in Secrets. Using public HTTPS URL.")

!rm -rf /content/Nano-U
!git clone {repo_url} /content/Nano-U
%cd /content/Nano-U
!pip install -q tf-keras tensorflow-model-optimization pyyaml

import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", gpus)
assert len(gpus) > 0, "ERROR: Connect to a GPU runtime before proceeding!"

### 2. Mount Google Drive & Extract Dataset

In [ ]:
# 2. Extract Dataset from Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile
zip_candidates = [
    "/content/drive/MyDrive/TinyAgri_full.zip",
    "/content/drive/MyDrive/thesis/TinyAgri_full.zip",
    "/content/TinyAgri_full.zip"
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)
assert zip_path is not None, f"Could not find TinyAgri_full.zip! Checked: {zip_candidates}"

print(f"Extracting {zip_path} into /content/Nano-U/data/...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/Nano-U/data/")
print("Extraction complete!")
!ls -la /content/Nano-U/data/TinyAgri_full/

In [ ]:
# 3. Generate 4-Fold Leave-One-Scene-Out (LOSO) Splits
!python scripts/prepare_cross_split.py

In [ ]:
# 4. Execute 4-Fold QAD Training Loop on GPU
import subprocess

for fold_idx in range(4):
    cfg = f"config/TinyAgri_fold{fold_idx}.yaml"
    print(f"\n{"="*55}\n  STARTING QAD FOR FOLD {fold_idx} ({cfg})\n{"="*55}")
    ret = subprocess.run(["python", "scripts/run_qad.py", "--config", cfg])
    if ret.returncode != 0:
        raise RuntimeError(f"Fold {fold_idx} failed!")
    print(f"✓ Fold {fold_idx} finished successfully!")

In [ ]:
# 5. Aggregate Summary and Package for Download
import os, json, subprocess
from google.colab import files

cv_summary = {}
for i in range(4):
    p = f"results/TinyAgri_fold{i}/pipeline_summary.json"
    if os.path.exists(p):
        with open(p) as f: cv_summary[f"fold_{i}"] = json.load(f)

with open("results/cv_4folds_summary.json", "w") as f:
    json.dump(cv_summary, f, indent=2)
print("✓ Saved results/cv_4folds_summary.json")

print("Compressing models and results into nanou_4folds_trained.zip...")
!zip -r -q nanou_4folds_trained.zip models results config
print(f"✓ Archive ready: nanou_4folds_trained.zip ({os.path.getsize("nanou_4folds_trained.zip") / 1e6:.2f} MB)")
files.download("nanou_4folds_trained.zip")